In [1]:
!pip install -q pyyaml h5py # Required to save models in HDF5 format

In [2]:
import os

import tensorflow as tf

In [3]:
print(tf.__version__)

2.2.0


#Loading a MNIST Dataset

In [5]:
(train_images, train_labels), (test_images, test_labels) = tf.keras.datasets.mnist.load_data()

#Using the first 1000 examples to speed up the process
train_labels = train_labels[:1000]
test_labels = test_labels[:1000]

train_images = train_images[:1000].reshape(-1, 28*28) / 255.0
test_images = test_images[:1000].reshape(-1, 28*28) / 255.0

#Define a Model


In [6]:
from tensorflow.keras import  Sequential
from tensorflow.keras.layers import Dense, Dropout

In [14]:
def create_model():
  model = Sequential()
  model.add(Dense(512, activation='elu', input_shape=(784,)))
  model.add(Dropout(0.5))
  model.add(Dense(10))

  model.compile(metrics = ['accuracy'],optimizer='adam', loss=tf.losses.SparseCategoricalCrossentropy(from_logits=True))

  return model



In [15]:
model = create_model()

In [16]:
model.summary()

Model: "sequential_3"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
dense_6 (Dense)              (None, 512)               401920    
_________________________________________________________________
dropout_3 (Dropout)          (None, 512)               0         
_________________________________________________________________
dense_7 (Dense)              (None, 10)                5130      
Total params: 407,050
Trainable params: 407,050
Non-trainable params: 0
_________________________________________________________________


#Saving the model during and after training


In [17]:
from tensorflow.keras.callbacks import ModelCheckpoint

In [18]:
checkpoint_path = 'training_1/cp.ckpt'
checkpoint_dir = os.path.dirname(checkpoint_path)

#Creating a call back that saves the model weights
cp_callback = ModelCheckpoint(filepath=checkpoint_path,
                              save_weights_only= True,
                              verbose=1)

model.fit(train_images, train_labels,
          epochs=10,
          validation_data= (test_images, test_labels),
          callbacks=[cp_callback])

Epoch 1/10
31/32 [============================>.] - ETA: 0s - loss: 1.1602 - accuracy: 0.6613
Epoch 00001: saving model to training_1/cp.ckpt
32/32 [==============================] - 0s 15ms/step - loss: 1.1573 - accuracy: 0.6620 - val_loss: 0.7438 - val_accuracy: 0.7600
Epoch 2/10
30/32 [===========================>..] - ETA: 0s - loss: 0.4773 - accuracy: 0.8562
Epoch 00002: saving model to training_1/cp.ckpt
32/32 [==============================] - 0s 9ms/step - loss: 0.4831 - accuracy: 0.8520 - val_loss: 0.6263 - val_accuracy: 0.7970
Epoch 3/10
26/32 [=======================>......] - ETA: 0s - loss: 0.3518 - accuracy: 0.8906
Epoch 00003: saving model to training_1/cp.ckpt
32/32 [==============================] - 0s 11ms/step - loss: 0.3551 - accuracy: 0.8890 - val_loss: 0.5023 - val_accuracy: 0.8330
Epoch 4/10
28/32 [=========================>....] - ETA: 0s - loss: 0.2782 - accuracy: 0.9230
Epoch 00004: saving model to training_1/cp.ckpt
32/32 [==============================] - 0s

In [19]:
ls {checkpoint_dir}


checkpoint  cp.ckpt.data-00000-of-00001  cp.ckpt.index


In [20]:
#Creating a new model with no weights
model = create_model()

loss, acc = model.evaluate(test_images, test_labels, verbose=2)
print("Untrained model, accuracy: {:5.2f}%".format(100*acc))

32/32 - 0s - loss: 2.4068 - accuracy: 0.0910
Untrained model, accuracy:  9.10%


In [21]:
model.load_weights(checkpoint_path)



In [22]:
loss, acc = model.evaluate(test_images, test_labels, verbose=2)
print("Untrained model, accuracy: {:5.2f}%".format(100*acc))

32/32 - 0s - loss: 0.4720 - accuracy: 0.8480
Untrained model, accuracy: 84.80%


#Saving the weights after every 5 epoochs

In [25]:
checkpoint_path = "training_2/cp-{epoch:04d}.ckpt"
checkpoint_dir = os.path.dirname(checkpoint_path)

cp_callback = ModelCheckpoint(filepath=checkpoint_path, verbose=1,
                              save_weights_only=True,
                              period=5)

model = create_model()

model.save_weights(checkpoint_path.format(epoch=0))

model.fit(train_images, train_labels,
              epochs = 50,
             callbacks= [cp_callback],
             validation_data = (test_images, test_labels),
             verbose=0)


Epoch 00005: saving model to training_2/cp-0005.ckpt

Epoch 00010: saving model to training_2/cp-0010.ckpt

Epoch 00015: saving model to training_2/cp-0015.ckpt

Epoch 00020: saving model to training_2/cp-0020.ckpt

Epoch 00025: saving model to training_2/cp-0025.ckpt

Epoch 00030: saving model to training_2/cp-0030.ckpt

Epoch 00035: saving model to training_2/cp-0035.ckpt

Epoch 00040: saving model to training_2/cp-0040.ckpt

Epoch 00045: saving model to training_2/cp-0045.ckpt

Epoch 00050: saving model to training_2/cp-0050.ckpt


In [26]:
ls {checkpoint_dir}


checkpoint                        cp-0025.ckpt.index
cp-0000.ckpt.data-00000-of-00001  cp-0030.ckpt.data-00000-of-00001
cp-0000.ckpt.index                cp-0030.ckpt.index
cp-0005.ckpt.data-00000-of-00001  cp-0035.ckpt.data-00000-of-00001
cp-0005.ckpt.index                cp-0035.ckpt.index
cp-0010.ckpt.data-00000-of-00001  cp-0040.ckpt.data-00000-of-00001
cp-0010.ckpt.index                cp-0040.ckpt.index
cp-0015.ckpt.data-00000-of-00001  cp-0045.ckpt.data-00000-of-00001
cp-0015.ckpt.index                cp-0045.ckpt.index
cp-0020.ckpt.data-00000-of-00001  cp-0050.ckpt.data-00000-of-00001
cp-0020.ckpt.index                cp-0050.ckpt.index
cp-0025.ckpt.data-00000-of-00001


In [27]:
latest = tf.train.latest_checkpoint(checkpoint_dir)
latest


'training_2/cp-0050.ckpt'

In [28]:
# Create a new model instance
model = create_model()

# Load the previously saved weights
model.load_weights(latest)

# Re-evaluate the model
loss, acc = model.evaluate(test_images,  test_labels, verbose=2)
print("Restored model, accuracy: {:5.2f}%".format(100*acc))


32/32 - 0s - loss: 0.7555 - accuracy: 0.8350
Restored model, accuracy: 83.50%


#Mannualy saving the weights

In [29]:
#Save the wights
model.save_weights('./checkpoints/my_checkpoint')

model = create_model()

model.load_weights('./checkpoints/my_checkpoint')

loss, acc = model.evaluate(test_images, test_labels, verbose=2)

print("Restored model, accuracy: {:5.2f}%".format(100*acc))



32/32 - 0s - loss: 0.7555 - accuracy: 0.8350
Restored model, accuracy: 83.50%


In [30]:
# Create and train a new model instance.
model = create_model()
model.fit(train_images, train_labels, epochs=5)

# Save the entire model as a SavedModel.
!mkdir -p saved_model
model.save('saved_model/my_model') 


Epoch 1/5
32/32 [==============================] - 0s 5ms/step - loss: 1.1876 - accuracy: 0.6210
Epoch 2/5
32/32 [==============================] - 0s 5ms/step - loss: 0.4785 - accuracy: 0.8520
Epoch 3/5
32/32 [==============================] - 0s 6ms/step - loss: 0.3499 - accuracy: 0.8910
Epoch 4/5
32/32 [==============================] - 0s 5ms/step - loss: 0.2970 - accuracy: 0.9150
Epoch 5/5
32/32 [==============================] - 0s 6ms/step - loss: 0.2420 - accuracy: 0.9350
Instructions for updating:
If using Keras pass *_constraint arguments to layers.
INFO:tensorflow:Assets written to: saved_model/my_model/assets


In [31]:
# my_model directory
!ls saved_model

# Contains an assets folder, saved_model.pb, and variables folder.
!ls saved_model/my_model


my_model
assets	saved_model.pb	variables


In [32]:
new_model = tf.keras.models.load_model('saved_model/my_model')

# Check its architecture
new_model.summary()


Model: "sequential_10"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
dense_20 (Dense)             (None, 512)               401920    
_________________________________________________________________
dropout_10 (Dropout)         (None, 512)               0         
_________________________________________________________________
dense_21 (Dense)             (None, 10)                5130      
Total params: 407,050
Trainable params: 407,050
Non-trainable params: 0
_________________________________________________________________


In [33]:
# Evaluate the restored model
loss, acc = new_model.evaluate(test_images,  test_labels, verbose=2)
print('Restored model, accuracy: {:5.2f}%'.format(100*acc))

print(new_model.predict(test_images).shape)


32/32 - 0s - loss: 0.4639 - accuracy: 0.8550
Restored model, accuracy: 85.50%
(1000, 10)


In [34]:
# Create and train a new model instance.
model = create_model()
model.fit(train_images, train_labels, epochs=5)

# Save the entire model to a HDF5 file.
# The '.h5' extension indicates that the model should be saved to HDF5.
model.save('my_model.h5') 


Epoch 1/5
32/32 [==============================] - 0s 6ms/step - loss: 1.1407 - accuracy: 0.6510
Epoch 2/5
32/32 [==============================] - 0s 5ms/step - loss: 0.4617 - accuracy: 0.8770
Epoch 3/5
32/32 [==============================] - 0s 5ms/step - loss: 0.3640 - accuracy: 0.8870
Epoch 4/5
32/32 [==============================] - 0s 6ms/step - loss: 0.2941 - accuracy: 0.9160
Epoch 5/5
32/32 [==============================] - 0s 6ms/step - loss: 0.2399 - accuracy: 0.9310


In [35]:
# Recreate the exact same model, including its weights and the optimizer
new_model = tf.keras.models.load_model('my_model.h5')

# Show the model architecture
new_model.summary()


Model: "sequential_11"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
dense_22 (Dense)             (None, 512)               401920    
_________________________________________________________________
dropout_11 (Dropout)         (None, 512)               0         
_________________________________________________________________
dense_23 (Dense)             (None, 10)                5130      
Total params: 407,050
Trainable params: 407,050
Non-trainable params: 0
_________________________________________________________________


In [36]:
loss, acc = new_model.evaluate(test_images,  test_labels, verbose=2)
print('Restored model, accuracy: {:5.2f}%'.format(100*acc))


32/32 - 0s - loss: 0.4797 - accuracy: 0.8390
Restored model, accuracy: 83.90%
